# Notebook 1 — Data Exploration & EDA

**Goal:** Understand the raw C-MAPSS dataset structure, sensor distributions,
operating condition regimes, and how degradation manifests in sensor signals.

**Dataset:** NASA Commercial Modular Aero-Propulsion System Simulation (C-MAPSS)
- 4 sub-datasets: FD001, FD002, FD003, FD004
- 21 sensors + 3 operational settings per cycle
- Run-to-failure trajectories for turbofan engines


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from src.data_loader import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor import add_piecewise_rul, find_constant_sensors

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)

datasets = load_all_datasets(data_dir='../data/raw')
print("Loaded datasets:", list(datasets.keys()))
datasets['FD001']['train'].head(3)


## 1.1 Dataset Summary Statistics

The four C-MAPSS datasets differ in number of engines, operating conditions,
and fault modes. Understanding these differences is essential before designing
a domain adaptation strategy.


In [ ]:
summary_rows = []
for ds_id, splits in datasets.items():
    df = splits['train']
    summary_rows.append({
        'Dataset':           ds_id,
        'Train Engines':     df['unit_id'].nunique(),
        'Test Engines':      splits['test']['unit_id'].nunique(),
        'Total Train Rows':  len(df),
        'Avg Cycles/Engine': round(df.groupby('unit_id')['cycle'].max().mean(), 1),
        'Min Cycles':        df.groupby('unit_id')['cycle'].max().min(),
        'Max Cycles':        df.groupby('unit_id')['cycle'].max().max(),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


**Insight:** FD002 and FD004 have substantially more engines (~260) because
they simulate 6 operating conditions. FD001 and FD003 operate under a single
sea-level condition with 100 engines each. Engine lifetimes vary between
~130 and ~370 cycles, with multi-condition datasets showing wider variance.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
for ax, ds_id in zip(axes, ['FD001', 'FD002', 'FD003', 'FD004']):
    life_lengths = datasets[ds_id]['train'].groupby('unit_id')['cycle'].max()
    ax.hist(life_lengths, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
    ax.set_title(f'{ds_id} Engine Lifetimes')
    ax.set_xlabel('Total Cycles')
    ax.set_ylabel('Count')
    ax.axvline(life_lengths.mean(), color='red', linestyle='--',
               label=f'Mean: {life_lengths.mean():.0f}')
    ax.legend(fontsize=8)
plt.suptitle('Distribution of Engine Lifetime Lengths Across Datasets', fontsize=13)
plt.tight_layout()
plt.show()


## 1.2 Identifying Constant Sensors Per Dataset

The paper notes that 7 sensors have constant readings in FD001/FD003.
These are RETAINED for cross-domain consistency — if they vary in other
datasets, removing them would break feature-space alignment.


In [ ]:
const_sensors_map = {}
for ds_id, splits in datasets.items():
    const = find_constant_sensors(splits['train'], SENSOR_COLS, threshold=1e-4)
    const_sensors_map[ds_id] = const
    print(f"{ds_id}: {len(const)} constant sensors → {const}")


In [ ]:
# Heatmap: which sensors are constant in which datasets
rows = []
for ds_id, const in const_sensors_map.items():
    row = {s: (1 if s in const else 0) for s in SENSOR_COLS}
    row['dataset'] = ds_id
    rows.append(row)
const_df = pd.DataFrame(rows).set_index('dataset')

fig, ax = plt.subplots(figsize=(18, 3))
sns.heatmap(const_df, cmap='RdYlGn_r', linewidths=0.5,
            annot=True, fmt='d', cbar=False, ax=ax)
ax.set_title('Constant Sensors per Dataset  (1 = constant, 0 = informative)')
ax.set_xlabel('Sensor')
plt.tight_layout()
plt.show()


**Insight:** Sensors 1, 5, 6, 10, 16, 18, and 19 are near-constant in
FD001 and FD003. However, they DO vary in FD002 and FD004. Retaining all
24 features maintains a consistent 24-dimensional input space across all
datasets — a prerequisite for the DANN architecture.


## 1.3 Sensor Distribution Comparison Across Datasets

The paper (Figure 3) shows normalised sensor distributions near failure.
Here we compare raw distributions across all four datasets.


In [ ]:
sensors_to_compare = ['sensor_2', 'sensor_7', 'sensor_11', 'sensor_12',
                       'sensor_14', 'sensor_15']

fig, axes = plt.subplots(len(sensors_to_compare), 1, figsize=(16, 3 * len(sensors_to_compare)))
colors = {'FD001': '#1f77b4', 'FD002': '#ff7f0e',
          'FD003': '#2ca02c', 'FD004': '#d62728'}

for ax, sensor in zip(axes, sensors_to_compare):
    for ds_id, splits in datasets.items():
        vals = splits['train'][sensor].dropna()
        ax.hist(vals, bins=60, alpha=0.5, label=ds_id,
                color=colors[ds_id], density=True)
    ax.set_title(f'{sensor} Value Distribution Across Datasets')
    ax.set_xlabel('Raw Sensor Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Raw Sensor Distributions — Cross-Dataset Comparison', fontsize=14)
plt.tight_layout()
plt.show()


**Insight:** FD001/FD003 (1 operating condition) show narrow, unimodal
distributions. FD002/FD004 (6 operating conditions) show multimodal
distributions reflecting the different operating regimes. This distribution
shift is exactly what the DANN model must overcome.


## 1.4 Average Sensor Trend vs. Remaining Useful Life (FD001)

Align all engines at their end-of-life point to visualise how sensors
behave as failure approaches.


In [ ]:
df_with_rul = add_piecewise_rul(datasets['FD001']['train'], max_rul=125)

sensors_to_plot = ['sensor_2', 'sensor_7', 'sensor_11',
                    'sensor_12', 'sensor_14', 'sensor_15']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, sensor in zip(axes.flatten(), sensors_to_plot):
    trend = df_with_rul.groupby('RUL')[sensor].mean()
    std   = df_with_rul.groupby('RUL')[sensor].std()
    ax.plot(trend.index, trend.values, color='steelblue', linewidth=2)
    ax.fill_between(trend.index,
                     trend.values - std.values,
                     trend.values + std.values,
                     alpha=0.15, color='steelblue')
    ax.invert_xaxis()
    ax.set_xlabel('Cycles Before Failure (RUL)')
    ax.set_ylabel('Mean Sensor Value')
    ax.set_title(f'{sensor} vs. RUL (FD001)')
    ax.grid(alpha=0.3)

plt.suptitle('Average Sensor Trends as Engines Approach Failure (FD001)', fontsize=14)
plt.tight_layout()
plt.show()


**Insight:** Sensors 11, 12, and 14 show clear monotonic trends as RUL
approaches zero — these are the primary degradation indicators for the
HPC (High Pressure Compressor) fault mode in FD001/FD002.
Sensor 7 and sensor 2 are noisier but still carry predictive signal.
This guides SHAP feature importance analysis in Notebook 07.


## 1.5 Operating Condition Clustering (FD002 / FD004)


In [ ]:
OP_COLS = ['op_setting_1', 'op_setting_2', 'op_setting_3']
df_fd002 = datasets['FD002']['train'].copy()

scaler  = StandardScaler()
op_sc   = scaler.fit_transform(df_fd002[OP_COLS])
kmeans  = KMeans(n_clusters=6, random_state=42, n_init=10)
df_fd002['op_cluster'] = kmeans.fit_predict(op_sc)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
scatter = axes[0].scatter(df_fd002['op_setting_1'], df_fd002['op_setting_2'],
                           c=df_fd002['op_cluster'], cmap='tab10', alpha=0.3, s=5)
plt.colorbar(scatter, ax=axes[0], label='Cluster')
axes[0].set_xlabel('op_setting_1 (Altitude proxy)')
axes[0].set_ylabel('op_setting_2 (Throttle Angle proxy)')
axes[0].set_title('FD002 Operating Condition Clusters (K=6)')

scatter2 = axes[1].scatter(df_fd002['op_setting_1'], df_fd002['op_setting_3'],
                            c=df_fd002['op_cluster'], cmap='tab10', alpha=0.3, s=5)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
axes[1].set_xlabel('op_setting_1')
axes[1].set_ylabel('op_setting_3 (Mach proxy)')
axes[1].set_title('FD002 — Setting 1 vs Setting 3')
plt.tight_layout()
plt.show()


**Insight:** The 6 operating conditions in FD002/FD004 form well-separated
clusters in the operational settings space. A model trained on FD001 (single
sea-level condition, bottom-left cluster only) will produce biased features
for the other 5 operating regimes — precisely what DANN addresses.


## 1.6 Per-Unit RUL Profile: Visualising Piecewise Linear Targets


In [ ]:
df_rul_plot = add_piecewise_rul(datasets['FD001']['train'], max_rul=125)
sample_units = [1, 5, 10, 20, 30, 50, 75, 100]

fig, ax = plt.subplots(figsize=(14, 6))
for unit in sample_units:
    ud = df_rul_plot[df_rul_plot['unit_id'] == unit]
    ax.plot(ud['cycle'], ud['RUL'], alpha=0.7, label=f'Unit {unit}')
ax.axhline(125, color='gray', linestyle=':', linewidth=1, label='RUL cap = 125')
ax.set_xlabel('Cycle')
ax.set_ylabel('RUL (piecewise linear, capped at 125)')
ax.set_title('Piecewise Linear RUL Labels — FD001 (Selected Engines)')
ax.legend(ncol=4, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Insight:** The piecewise linear target treats all engines as equally healthy
during the flat phase (RUL = 125) then assigns linearly decreasing RUL.
The flat region length varies per unit because some engines start degrading
earlier than others, but all degrade monotonically once past the inflection point.
